In [1]:
# ============================================
# Blockwise permutation + Areal GP + VI module
# Using meuse_obs.zip (geopandas) & elev as X
# ============================================

import os
from typing import Dict, Any, Sequence

import numpy as np
import torch
import torch.optim as optim
from tqdm import tqdm

import pandas as pd
import geopandas as gpd  # <--- NEW

from GPModel import GPModel
from GPArealModel import GPArealModel
from VIGP_Unlinked import VIGP_Unlinked

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _to_tensor(x, dtype=torch.float32):
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)
    return torch.as_tensor(x, device=device, dtype=dtype)


# ---------------------------------------------------------
# STEP 2 helper: construct blockwise permutation at data level
# ---------------------------------------------------------

def make_blockwise_permuted_data(
    coords,
    X,
    Y,
    n_blocks: int,
    n_locations: int,
    seed: int = 521,
) -> Dict[str, torch.Tensor]:
    """
    Prepare original (restricted) and blockwise-permuted data.

    Inputs
    ------
    coords : array-like (N, d)
    X      : array-like (N,) or (N, 1)
    Y      : array-like (N,)
    n_blocks, n_locations : define N_use = n_blocks * n_locations
    seed : for reproducible permutations

    Returns a dict with:
        coords_orig, X_orig, Y_orig   : restricted, sorted (unpermuted)
        coords_perm, X_perm, Y_perm   : permuted inside blocks
        region_assignments            : (N_use,) with values 0,...,B-1
        perm_matrix_x, perm_matrix_s  : (N_use, N_use) block-diagonal perms
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    coords = _to_tensor(coords)
    Y = _to_tensor(Y).view(-1)  # (N,)
    X = _to_tensor(X)

    if X.ndim == 1:
        X = X.unsqueeze(1)  # (N, 1)

    N_total = coords.shape[0]
    N_use = n_blocks * n_locations
    if N_total < N_use:
        raise ValueError(f"Not enough locations: N={N_total}, required N_use={N_use}.")

    # Sort by first coordinate (x), then restrict to N_use
    sort_idx = torch.argsort(coords[:, 0])
    sort_idx = sort_idx[:N_use]

    coords_orig = coords[sort_idx].contiguous()  # (N_use, d)
    X_orig = X[sort_idx].contiguous()            # (N_use, 1)
    Y_orig = Y[sort_idx].contiguous()            # (N_use,)

    N = N_use

    # Region assignments: fixed blocks
    region_assignments = torch.zeros(N, dtype=torch.long, device=device)
    for b in range(n_blocks):
        start = b * n_locations
        end = (b + 1) * n_locations
        region_assignments[start:end] = b

    # Blockwise permutation matrices
    #   - perm_matrix_x: permutes X,Y jointly within block
    #   - perm_matrix_s: independent perm for coordinates
    perm_x = torch.randperm(n_locations)
    perm_s = torch.randperm(n_locations)
    # Convert perm_s into a permutation matrix
    perm_matrix_s = torch.zeros(n_locations, n_locations, dtype=torch.float32, device=device)
    perm_matrix_s[torch.arange(n_locations), perm_s] = 1

    # Convert perm_x into a permutation matrix
    perm_matrix_x = torch.zeros(n_locations, n_locations, dtype=torch.float32, device=device)
    perm_matrix_x[torch.arange(n_locations), perm_x] = 1
    
    unique_regions = torch.unique(region_assignments)
    X_perm = torch.zeros_like(X_orig)
    coords_perm = torch.zeros_like(coords_orig)
    for i, region in enumerate(unique_regions):
            # Get indices for the current region
            indices = torch.where(region_assignments == region)[0]
            
            # Assign the shuffled values back
            X_perm[indices] = X_orig[indices[perm_x]]
            coords_perm[indices] = coords_orig[indices[perm_s]]

    
    # Apply permutations
    Y_perm = Y_orig

    return {
        "coords_orig": coords_orig,
        "X_orig": X_orig,
        "Y_orig": Y_orig,
        "coords_perm": coords_perm,
        "X_perm": X_perm,
        "Y_perm": Y_perm,
        "region_assignments": region_assignments,
        "perm_matrix_x": perm_matrix_x,
        "perm_matrix_s": perm_matrix_s,
    }


# ---------------------------------------------------------
# STEP 3a: Areal GP ONLY (disentangled)
# ---------------------------------------------------------

def run_areal_gp(
    coords_perm: torch.Tensor,
    X_perm: torch.Tensor,
    Y_perm: torch.Tensor,
    region_assignments: torch.Tensor,
    n_blocks: int,
    n_locations: int,
    seed: int = 521,
    niter_GPAreal: int = 3000,
) -> Dict[str, Any]:
    """
    Run GPArealModel (areal GP) on block-averaged data,
    using permuted coordinates and block structure.
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    coords_perm = _to_tensor(coords_perm)
    Y_perm = _to_tensor(Y_perm).view(-1)
    X_perm = _to_tensor(X_perm)

    if X_perm.ndim == 1:
        X_perm = X_perm.unsqueeze(1)
    p = X_perm.shape[1]
    if p != 1:
        raise ValueError(f"GPArealModel wrapper assumes scalar X (p=1), got p={p}.")

    region_assignments = region_assignments.to(device=device, dtype=torch.long)

    N = coords_perm.shape[0]
    assert N == n_blocks * n_locations, "N must equal n_blocks * n_locations."

    # Block averages of X and Y
    ybar = torch.zeros(n_blocks, device=device)
    xbar = torch.zeros(n_blocks, p, device=device)

    for b in range(n_blocks):
        start = b * n_locations
        end = (b + 1) * n_locations
        idx = torch.arange(start, end, device=device)
        ybar[b] = Y_perm[idx].mean()
        xbar[b] = X_perm[idx].mean(dim=0)

    gpa = GPArealModel().to(device)
    opt_gpa = optim.AdamW(gpa.parameters(), lr=0.01, weight_decay=0.01)

    for _ in tqdm(range(niter_GPAreal), desc="Train GPArealModel (areal)"):
        opt_gpa.zero_grad()
        loss = gpa(coords_perm, region_assignments, xbar, ybar)
        loss.backward()
        opt_gpa.step()
        # with torch.no_grad():
        #     gpa.sigmasq.clamp_(min=1e-6)
        #     gpa.phi.clamp_(min=1e-6)
        #     gpa.tausq.clamp_(min=1e-6)

    return {
        "nu": float(gpa.nu.item()),
        "phi": float(np.exp(gpa.logphi.item())),
        "sigmasq": float(np.exp(gpa.logsigmasq.item())),
        "tausq": float(np.exp(gpa.logtausq.item())),
        "beta": gpa.beta.detach().cpu().numpy(),
    }


# ---------------------------------------------------------
# STEP 3b: VI for Unlinked GP ONLY (disentangled)
# ---------------------------------------------------------

def run_vi_unlinked(
    coords_perm: torch.Tensor,
    X_perm: torch.Tensor,
    Y_perm: torch.Tensor,
    n_blocks: int,
    n_locations: int,
    seed: int = 521,
    niter_VI: int = 100,
    phi_prior_ub: float = 0.5,
    phi_prior_lb: float = 0.0,
    lr_piS=0.01,
    lr_piX=0.01,
    tau_grid: Sequence[float] = (0.2, 0.4, 0.6, 0.8, 0.9),
    VX_ub: float = 0.5,
    VS_ub: float = 0.5,
    pi_X_true = None,
    pi_S_true = None
) -> Dict[str, Any]:
    """
    Run VIGP_Unlinked (VI for unlinked GP) on blockwise permuted X,Y,S.

    Returns a dict with VI outputs for each tau:
        {
          "by_tau": { tau_value: vi_out, ... },
          "taus": [...],
          "prior_parameters": {...}
        }
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    coords_perm = _to_tensor(coords_perm)
    Y_perm = _to_tensor(Y_perm).view(-1)  # (N,)
    X_perm = _to_tensor(X_perm)

    if X_perm.ndim == 1:
        X_perm = X_perm.unsqueeze(1)
    p = X_perm.shape[1]
    if p != 1:
        raise ValueError(f"VI wrapper assumes scalar X (p=1), got p={p}.")



    N = coords_perm.shape[0]
    assert N == n_blocks * n_locations, "N must equal n_blocks * n_locations."

    # Distances from permuted coordinates
    Dist = torch.cdist(coords_perm, coords_perm, p=2)
    Dist = (Dist + Dist.T) / 2.0

    # Block-shaped tensors for VI
    X_blocks = X_perm.view(n_blocks, n_locations)  # (B, n_i)
    Y_blocks = Y_perm.view(n_blocks, n_locations)  # (B, n_i)

    n_steps = 50
    n_phi_samples = 200
    n_piX_sample = 50
    n_piS_sample = 50

    prior_parameters = {
        "a1": 0.1,
        "b1": 0.1,
        "a2": 0.1,
        "b2": 0.1,
        "eta_X_sq": 0.1,
        "eta_S_sq": 0.1,
        "mu_beta": 0.0,
        "sigmasq_beta": 100.0,
        "phi_prior_lb": phi_prior_lb,
        "phi_prior_ub": phi_prior_ub,
    }


    vi_results_by_tau: Dict[float, Any] = {}

    for tau in tau_grid:
        vi_out = VIGP_Unlinked(
            n_iter=niter_VI,
            n_blocks=n_blocks,
            n_locations=n_locations,
            X=X_blocks,
            Y=Y_blocks,
            Dist=Dist,
            n_steps=n_steps,
            n_phi_samples=n_phi_samples,
            n_piX_sample=n_piX_sample,
            tau_X=tau,
            tau_S=tau,
            n_piS_sample=n_piS_sample,
            seed=seed,
            fix_piX=False,
            fix_piS=False,
            fix_mu_lambda_beta=False,
            fix_sigmasq_lambda_beta=False,
            fix_lambda_b1=False,
            lambda_b1_fixed=((n_blocks * n_locations) * 0.5 + 0.1) * 5,
            fix_lambda_b2=False,
            M_X_star_fixed=None,
            M_S_star_fixed= None,
            V_X_star_fixed=torch.eye(n_locations, n_locations, device=device),
            V_S_star_fixed=torch.eye(n_locations, n_locations, device=device),
            phi_init = 0.05,
            mean_Rphi_inv_fixed=None,
            fix_mean_Rphi_inv=False,
            pi_X_true=pi_X_true,
            pi_S_true=pi_S_true,
            VX_ub=VX_ub,
            VS_ub=VS_ub,
            lr_piS=lr_piS,
            lr_piX=lr_piX,
            prior_parameters=prior_parameters,
        )
        vi_results_by_tau[float(tau)] = vi_out

    return {
        "by_tau": vi_results_by_tau,
        "taus": list(map(float, tau_grid)),
        "prior_parameters": prior_parameters,
    }


# ============================================================
# Demo: FULL PIPELINE on meuse_obs.zip
# ============================================================


# __file__ is not defined in interactive environments (e.g., Jupyter notebooks).
# Fall back to the current working directory when needed.
try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

MEUSE_OBS_PATH = os.path.abspath(os.path.join(
    base_dir,
    "..",
    "data", "data_analysis",
    "meuse_obs.zip",
))

meuse_obs = gpd.read_file(MEUSE_OBS_PATH)

# coords from geometry (x, y)
coords_meuse = np.column_stack(
    [meuse_obs.geometry.x.to_numpy(), meuse_obs.geometry.y.to_numpy()]
)

# response: log1p(zinc)
Y_meuse = np.log1p(meuse_obs["zinc"].to_numpy())

# scalar covariate: elevation
# (column name in this dataset is 'elev'; if it's 'elevation' on your side, change here)
#X_meuse = np.sqrt(meuse_obs["dist"].to_numpy())
X_meuse = meuse_obs["elev"].to_numpy()

# center covariate and response to zero mean
X_meuse_mean = X_meuse.mean()
Y_meuse_mean = Y_meuse.mean()

X_meuse = X_meuse - X_meuse_mean
Y_meuse = Y_meuse - Y_meuse_mean

print(f"Centered X_meuse mean: {X_meuse.mean():.6f}, Y_meuse mean: {Y_meuse.mean():.6f}")

N_meuse = coords_meuse.shape[0]
print(f"Loaded meuse_obs: N = {N_meuse}")

# Rescale coordinates for numerical stability (e.g. to km)
coords_center = coords_meuse.mean(axis=0)
coords_scaled = (coords_meuse) / 1000.0  # now roughly in km
# Normalize coordinates to [0, 1] range
coords_min = coords_meuse.min(axis=0)
coords_max = coords_meuse.max(axis=0)
coords_scaled = (coords_meuse - coords_min) / (coords_max - coords_min)

print(f"Coordinates normalized to [0, 1]: min={coords_scaled.min():.6f}, max={coords_scaled.max():.6f}")


Centered X_meuse mean: -0.000000, Y_meuse mean: 0.000000
Loaded meuse_obs: N = 155
Coordinates normalized to [0, 1]: min=0.000000, max=1.000000


In [2]:
coords_perm_data = make_blockwise_permuted_data(
    coords=coords_scaled,
    X=X_meuse,
    Y=Y_meuse,
    n_blocks=31,
    n_locations=5,
    seed=2029,
)
coords_perm = coords_perm_data["coords_perm"]
X_perm = coords_perm_data["X_perm"]
Y_perm = coords_perm_data["Y_perm"]
region_assignments = coords_perm_data["region_assignments"]
perm_matrix_x = coords_perm_data["perm_matrix_x"]
perm_matrix_s = coords_perm_data["perm_matrix_s"]


In [3]:

# -----------------------------
# STEP 3a: Areal GP on permuted data
# -----------------------------
areal_results = run_areal_gp(
    coords_perm=coords_perm,
    X_perm=X_perm,
    Y_perm=Y_perm,
    region_assignments=region_assignments,
    n_blocks=31,
    n_locations=5,
    seed=2030,
    niter_GPAreal=1000,   # shorter for demo
)


Train GPArealModel (areal):   0%|          | 0/1000 [00:00<?, ?it/s]/Users/debangandey/Documents/GitHub/SpatialReg-Unlinked/src/GPArealModel.py:25: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729646995093/work/aten/src/ATen/native/TensorShape.cpp:3687.)
  quadratic_term = residual.T @ torch.linalg.solve(K_NN_noise, residual)
Train GPArealModel (areal): 100%|██████████| 1000/1000 [00:00<00:00, 1999.84it/s]


In [4]:
areal_results

{'nu': 0.5,
 'phi': 0.11086477588883815,
 'sigmasq': 0.1933048393944298,
 'tausq': 0.04657801879718212,
 'beta': array([-0.3482511], dtype=float32)}

In [6]:
# -----------------------------
# STEP 3b: VI unlinked on permuted data
# -----------------------------
vi_results = run_vi_unlinked(
    coords_perm=coords_perm,
    X_perm=X_perm,
    Y_perm=Y_perm,
    n_blocks=31,
    n_locations=5,
    seed=2025,
    niter_VI=100,
    phi_prior_lb=0.01, 
    phi_prior_ub=1.3,
    pi_X_true= perm_matrix_x.T,
    pi_S_true = perm_matrix_s.T, lr_piS = 0.001, 
    tau_grid= (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9), VX_ub=0.5, VS_ub=0.5
) 


  0%|          | 0/100 [00:00<?, ?it/s]

Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: -9.5963e-08
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: -9.5963e-08


  1%|          | 1/100 [00:20<33:57, 20.58s/it]

Iter 1/100 | mu_lambda_beta: -0.0927 | 
 sigmasq_lambda_beta: 0.0000 | 
 lambda_a1: 77.6000 | lambda_b1: 138.1774 | lambda_a2: 77.6000 | lambda_b2: 116.7562
‣  E[1/ϕ]: 11.5455 | ‣ E[Sigmasq/ϕ]: 20.8268 | ‣ ||mu_W||: 2.3081
Number of correct permutations recognized for piX: 5.0
Number of correct permutations recognized for piS: 1.0
Total Loss: 1.0142
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 4.6646e-03
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 3.3653e-03


  2%|▏         | 2/100 [00:41<33:41, 20.62s/it]

...

Iter 26/100 | mu_lambda_beta: -0.2250 | 
 sigmasq_lambda_beta: 0.0010 | 
 lambda_a1: 77.6000 | lambda_b1: 58.6754 | lambda_a2: 77.6000 | lambda_b2: 16.2502
‣  E[1/ϕ]: 0.9584 | ‣ E[Sigmasq/ϕ]: 0.7341 | ‣ ||mu_W||: 4.6213
Number of correct permutations recognized for piX: 5.0
Number of correct permutations recognized for piS: 5.0
Total Loss: 0.4561
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 1.8633e-01
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 6.6972e-02


 27%|██▋       | 27/100 [09:12<25:08, 20.67s/it]

Iter 27/100 | mu_lambda_beta: -0.2236 | 
 sigmasq_lambda_beta: 0.0010 | 
 lambda_a1: 77.6000 | lambda_b1: 58.8080 | lambda_a2: 77.6000 | lambda_b2: 16.2187
‣  E[1/ϕ]: 0.9633 | ‣ E[Sigmasq/ϕ]: 0.7395 | ‣ ||mu_W||: 4.6259
Number of correct permutations recognized for piX: 5.0
Number of correct permutations recognized for piS: 5.0
Total Loss: 0.4558
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 1.8751e-01
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 6.8312e-02


 27%|██▋       | 27/100 [09:14<25:00, 20.55s/it]


KeyboardInterrupt: 

In [10]:
coords_orig = coords_perm_data["coords_orig"]
X_orig = coords_perm_data["X_orig"]
Y_orig = coords_perm_data["Y_orig"]
# -----------------------------
# STEP 4: Oracle GP on original data
# -----------------------------
gp_oracle = GPModel().to(device)
opt_gp = optim.AdamW(gp_oracle.parameters(), lr=0.001, weight_decay=0.01)

for _ in tqdm(range(20000), desc="Train GPModel (oracle, meuse)"):
    opt_gp.zero_grad()
    loss = gp_oracle(coords_orig, X_orig, Y_orig)
    loss.backward()
    opt_gp.step()
    # with torch.no_grad():
    #     gp_oracle.sigmasq.clamp_(min=1e-6)
    #     gp_oracle.phi.clamp_(min=1e-6)
    #     gp_oracle.tausq.clamp_(min=1e-6)

oracle_params = {
    "nu": float(gp_oracle.nu.item()),
    "phi": float(np.exp(gp_oracle.logphi.item())),
    "sigmasq": float(np.exp(gp_oracle.logsigmasq.item())),
    "tausq": float(np.exp(gp_oracle.logtausq.item())),
    "beta": gp_oracle.beta.detach().cpu().numpy(),
}


Train GPModel (oracle, meuse): 100%|██████████| 20000/20000 [00:50<00:00, 399.61it/s]


In [11]:
print(f"Oracle sigmasq/phi: {oracle_params['sigmasq'] / oracle_params['phi']:.6f}")
print("\nOracle GP parameters:")
for key, value in oracle_params.items():
    print(f"  {key}: {value}")

Oracle sigmasq/phi: 2.464693

Oracle GP parameters:
  nu: 0.5
  phi: 0.37909581630526806
  sigmasq: 0.9343548422249289
  tausq: 0.0009767571472137971
  beta: [-0.2759907]
